# OmicSieve K-Sweep Comparison (Unsupervised Pipeline + Compact Baselines)

In [1]:
from pathlib import Path
from functools import partial
import re

import warnings
warnings.filterwarnings('ignore')

import os
import time
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedShuffleSplit, StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, balanced_accuracy_score, silhouette_score
from sklearn.cluster import KMeans
from sklearn.decomposition import KernelPCA
from sklearn.feature_selection import SelectKBest, f_classif, mutual_info_classif

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from torch.optim.lr_scheduler import ReduceLROnPlateau

import xgboost as xgb
from scipy.stats import spearmanr

import plotly.graph_objects as go

# Set reproducibility seeds
os.environ['PYTHONHASHSEED'] = '42'
np.random.seed(42)
random.seed(42)
torch.manual_seed(42)

# Reproducible torch behavior where possible
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
torch.use_deterministic_algorithms(True)

In [2]:
# Top-level task switch
# Choose one: 'grade' or 'tp53'
task = 'grade'  # or 'tp53'

# K sweep values for both OmicSieve components and baseline selected features
K_SWEEP_VALUES = [10, 20, 30, 50, 80, 100]

# Keep final figure focused on hold-out balanced accuracy by default
INCLUDE_CV_CURVES = False

# Only this final figure will be saved
FINAL_FIG_PATH = Path(f'{task}_k_sweep_balanced_accuracy_plotly.html')

print(f'Active task: {task}')
print(f'K sweep: {K_SWEEP_VALUES}')

Active task: grade
K sweep: [10, 20, 30, 50, 80, 100]


In [3]:
# Load the raw tables
DATA_DIR = Path('/media/storage/Amir/Dataset/Deep Mut/')
MUT_INFO_PATH = DATA_DIR / 'mut_info.tsv'
RNASEQ_PATH = DATA_DIR / 'rnaseq_data.tsv'
META_PATH = DATA_DIR / 'meta_data.tsv'

meta_df = pd.read_csv(META_PATH, sep='	')
header_df = pd.read_csv(RNASEQ_PATH, sep='	', header=None, nrows=1, dtype=str)
gene_names = header_df.iloc[0, :].tolist()
rnaseq_df = pd.read_csv(RNASEQ_PATH, sep='	', header=None, skiprows=1, dtype=str, engine='python', on_bad_lines='warn')

if rnaseq_df.shape[1] > len(gene_names):
    rnaseq_df = rnaseq_df.iloc[:, :len(gene_names)]
elif rnaseq_df.shape[1] < len(gene_names):
    for _ in range(len(gene_names) - rnaseq_df.shape[1]):
        rnaseq_df[len(rnaseq_df.columns)] = None

rnaseq_df.columns = gene_names
rnaseq_df = rnaseq_df.rename(columns={gene_names[0]: 'sample_id'})

print(f'RNA-seq shape: {rnaseq_df.shape}')

RNA-seq shape: (9626, 19311)


In [4]:
TCGA_ID_PATTERN = re.compile(r'TCGA-[A-Z0-9]+-[A-Z0-9]+-\d+')


def _clean_meta(df: pd.DataFrame) -> pd.DataFrame:
    meta = df.copy()
    meta.columns = (
        meta.columns.astype(str)
        .str.strip()
        .str.strip('"')
        .str.lower()
        .str.replace(' ', '_', regex=False)
    )
    if 'sample' in meta.columns:
        meta['patient_id'] = meta['sample'].astype(str).str.strip().str.replace('"', '')
    meta = meta.dropna(subset=['patient_id']).drop_duplicates('patient_id')
    if 'histological_grade' in meta.columns:
        meta['histological_grade'] = (
            meta['histological_grade'].astype(str).str.strip().str.upper().replace({'': pd.NA})
        )
    return meta


def _clean_rnaseq(df: pd.DataFrame) -> pd.DataFrame:
    gnames = df.columns.astype(str).str.strip().str.replace('"', '')
    sample_ids = df.iloc[:, 0].astype(str).str.strip().str.replace('"', '')
    expr_data = df.iloc[:, 1:].copy()
    expr_data.columns = gnames[:len(expr_data.columns)]
    expr_data = expr_data.apply(pd.to_numeric, errors='coerce')

    rna_clean = pd.concat([sample_ids.reset_index(drop=True), expr_data.reset_index(drop=True)], axis=1)
    rna_clean.columns = ['patient_id'] + list(expr_data.columns)
    rna_clean = rna_clean[rna_clean['patient_id'].str.len() > 0].dropna(subset=['patient_id'])
    rna_clean = rna_clean.set_index('patient_id')

    rna_final = rna_clean.groupby(level=0).mean()
    rna_final = rna_final.T
    rna_final['gene_id'] = rna_final.index
    rna_final = rna_final[['gene_id'] + [c for c in rna_final.columns if c != 'gene_id']]
    rna_final = rna_final.reset_index(drop=True)
    return rna_final


def grade_to_risk_label(grade):
    low_risk_grades = {'G1', 'LOW GRADE'}
    high_risk_grades = {'G3', 'G4', 'HIGH GRADE'}
    if pd.isna(grade):
        return pd.NA
    g = str(grade).strip().upper()
    if g in low_risk_grades:
        return 0
    if g in high_risk_grades:
        return 1
    return pd.NA


def build_task_labeled_data(task_name, meta_clean, rnaseq_clean, mut_info_path):
    task_name = str(task_name).strip().lower()

    rnaseq_patient_ids = set(rnaseq_clean.columns) - {'gene_id'}
    matched_patient_ids = sorted(set(meta_clean['patient_id']) & rnaseq_patient_ids)

    meta_task = meta_clean[meta_clean['patient_id'].isin(matched_patient_ids)].copy().reset_index(drop=True)
    rnaseq_task = rnaseq_clean[['gene_id'] + matched_patient_ids].copy().reset_index(drop=True)

    print('=== DATASET BEFORE LABELING ===')
    print(f'Metadata shape: {meta_task.shape}')
    print(f'RNA-seq shape: {rnaseq_task.shape}')

    if task_name == 'grade':
        meta_task['label'] = meta_task['histological_grade'].apply(grade_to_risk_label)
        valid_idx = meta_task['label'].notna()
        labeled_meta_df = meta_task[valid_idx].copy().reset_index(drop=True)
        labeled_meta_df['label'] = labeled_meta_df['label'].astype(int)

        ordered_patient_ids = sorted(set(labeled_meta_df['patient_id']) & rnaseq_patient_ids)
        labeled_meta_df = labeled_meta_df.set_index('patient_id').loc[ordered_patient_ids].reset_index()
        labeled_rnaseq_df = rnaseq_task[['gene_id'] + ordered_patient_ids].reset_index(drop=True)

        print()
        print('=== DATASET AFTER LABELING ===')
        print(f'Metadata shape: {labeled_meta_df.shape}')
        print(f'RNA-seq shape: {labeled_rnaseq_df.shape}')
        print()
        print('Risk label distribution:')
        print(labeled_meta_df['label'].value_counts().sort_index())
        print(f"  0 (Low risk):  {(labeled_meta_df['label'] == 0).sum()}")
        print(f"  1 (High risk): {(labeled_meta_df['label'] == 1).sum()}")

    elif task_name == 'tp53':
        mut_df = pd.read_csv(mut_info_path, sep='	', dtype=str)
        mut_df.columns = (
            mut_df.columns.astype(str)
            .str.strip()
            .str.strip('"')
            .str.lower()
            .str.replace(' ', '_', regex=False)
        )

        index_ids = pd.Index(mut_df.index).astype(str)
        index_tcga_fraction = index_ids.str.match(r'^TCGA-[A-Z0-9]+-[A-Z0-9]+-\d+').mean()
        if index_tcga_fraction > 0.5:
            mut_df = mut_df.copy()
            mut_df['patient_id'] = index_ids.str.strip().str.replace('"', '')
            id_col = 'patient_id'
        else:
            id_candidates = ['patient_id', 'sample', 'x_patient', 'rnaseqid', 'sample_id', 'mutid']
            id_col = next((c for c in id_candidates if c in mut_df.columns), mut_df.columns[0])
            mut_df[id_col] = mut_df[id_col].astype(str).str.strip().str.replace('"', '')

        mut_df = mut_df.dropna(subset=[id_col]).drop_duplicates(id_col)
        mutation_cols = [c for c in mut_df.columns if c != id_col]
        tp53_col = next((c for c in mutation_cols if 'tp53' in c.lower()), None)
        if tp53_col is None:
            raise ValueError('Could not locate a TP53 column in mut_info.tsv after preprocessing.')

        binary_mut_info = mut_df[[id_col, tp53_col]].copy()
        binary_mut_info['label'] = binary_mut_info[tp53_col].apply(
            lambda x: 0 if str(x).strip().lower() == 'wild' else 1
        )

        mutation_by_patient = binary_mut_info.set_index(id_col)['label']
        meta_task['label'] = meta_task['patient_id'].map(mutation_by_patient)

        valid_idx = meta_task['label'].notna()
        labeled_meta_df = meta_task[valid_idx].copy().reset_index(drop=True)
        labeled_meta_df['label'] = labeled_meta_df['label'].astype(int)

        ordered_patient_ids = sorted(set(labeled_meta_df['patient_id']) & rnaseq_patient_ids)
        labeled_meta_df = labeled_meta_df.set_index('patient_id').loc[ordered_patient_ids].reset_index()
        labeled_rnaseq_df = rnaseq_task[['gene_id'] + ordered_patient_ids].reset_index(drop=True)

        print()
        print('=== DATASET AFTER LABELING ===')
        print(f'Found TP53 column: {tp53_col}')
        print(f'Metadata shape: {labeled_meta_df.shape}')
        print(f'RNA-seq shape: {labeled_rnaseq_df.shape}')
        print()
        print('Mutation label distribution:')
        print(labeled_meta_df['label'].value_counts().sort_index())
        print(f"  0 (Wild-type): {(labeled_meta_df['label'] == 0).sum()}")
        print(f"  1 (Mutated):   {(labeled_meta_df['label'] == 1).sum()}")

    else:
        raise ValueError("task must be one of: 'grade', 'tp53'")

    X_raw = labeled_rnaseq_df.drop(columns='gene_id').T.to_numpy(dtype=float)
    y_binary = labeled_meta_df['label'].astype(int).to_numpy()
    feature_names = labeled_rnaseq_df['gene_id'].astype(str).tolist()

    assert X_raw.shape[0] == y_binary.shape[0]

    return {
        'labeled_meta_df': labeled_meta_df,
        'labeled_rnaseq_df': labeled_rnaseq_df,
        'X_raw': X_raw,
        'y_binary': y_binary,
        'feature_names': feature_names,
    }


clean_meta_df = _clean_meta(meta_df)
clean_rnaseq_df = _clean_rnaseq(rnaseq_df)

task_data = build_task_labeled_data(task, clean_meta_df, clean_rnaseq_df, MUT_INFO_PATH)
X_raw = task_data['X_raw']
y_binary = task_data['y_binary']
feature_names = task_data['feature_names']

print(f'Feature count before selection: {len(feature_names)}')

=== DATASET BEFORE LABELING ===
Metadata shape: (9626, 41)
RNA-seq shape: (19310, 9627)

=== DATASET AFTER LABELING ===
Metadata shape: (2494, 42)
RNA-seq shape: (19310, 2495)

Risk label distribution:
label
0     312
1    2182
Name: count, dtype: int64
  0 (Low risk):  312
  1 (High risk): 2182
Feature count before selection: 19310


### Using KernelPCA + Neural Feature Selector + XGBoost

#### =============================================================================
#### STAGE 0: Data Preparation
#### =============================================================================

In [5]:
def stage_0_prepare_data(X, y, test_size=0.15, val_size=0.15, random_state=42):
    # Stratified split: train+val vs test
    splitter1 = StratifiedShuffleSplit(n_splits=1, test_size=test_size, random_state=random_state)
    train_val_idx, test_idx = next(splitter1.split(X, y))

    X_train_val = X[train_val_idx]
    y_train_val = y[train_val_idx]
    X_test = X[test_idx]
    y_test = y[test_idx]

    # Stratified split: train vs val
    val_size_adjusted = val_size / (1.0 - test_size)
    splitter2 = StratifiedShuffleSplit(n_splits=1, test_size=val_size_adjusted, random_state=random_state)
    train_idx, val_idx = next(splitter2.split(X_train_val, y_train_val))

    X_train = X_train_val[train_idx]
    y_train = y_train_val[train_idx]
    X_val = X_train_val[val_idx]
    y_val = y_train_val[val_idx]

    # Fit scaler on train, apply to all
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_val_scaled = scaler.transform(X_val)
    X_test_scaled = scaler.transform(X_test)

    result = {
        'X_train': X_train_scaled,
        'y_train': y_train,
        'X_val': X_val_scaled,
        'y_val': y_val,
        'X_test': X_test_scaled,
        'y_test': y_test,
        'scaler': scaler,
        'n_features': X.shape[1],
    }

    print('STAGE 0: Data Prepared')
    print(f"  Train: {result['X_train'].shape} | Val: {result['X_val'].shape} | Test: {result['X_test'].shape}")
    print('  Label distribution:')
    print(f"    Train - Class0: {(y_train==0).sum()}, Class1: {(y_train==1).sum()}")
    print(f"    Val   - Class0: {(y_val==0).sum()}, Class1: {(y_val==1).sum()}")
    print(f"    Test  - Class0: {(y_test==0).sum()}, Class1: {(y_test==1).sum()}")

    return result


data = stage_0_prepare_data(X_raw, y_binary, random_state=42)

STAGE 0: Data Prepared
  Train: (1745, 19310) | Val: (374, 19310) | Test: (375, 19310)
  Label distribution:
    Train - Class0: 218, Class1: 1527
    Val   - Class0: 47, Class1: 327
    Test  - Class0: 47, Class1: 328


#### =============================================================================
#### STAGE 1: Unsupervised Components (KernelPCA)
#### =============================================================================

In [6]:
def stage_1_unsupervised_components(X_train, X_val, method='kpca', n_components=100, kernel='rbf'):
    method = str(method).lower().strip()

    print(f'  [{method.upper()}] X_train shape: {X_train.shape}')

    t0 = time.time()
    if method == 'kpca':
        component_op = Pipeline([
            ('scaler', StandardScaler()),
            ('kpca', KernelPCA(n_components=n_components, kernel=kernel, fit_inverse_transform=False)),
        ])
        X_train_components = component_op.fit_transform(X_train)
        X_val_components = component_op.transform(X_val)
        component_name = f'KPCA_{kernel.upper()}'
    else:
        raise ValueError(f'Unknown method: {method}. Use kpca')

    train_time = time.time() - t0
    infer_per_sample_ms = (train_time / max(len(X_train) + len(X_val), 1)) * 1000

    print(f'STAGE 1: {component_name} fitted')
    print(f'  X_train_components: {X_train_components.shape}')
    print(f'  X_val_components: {X_val_components.shape}')
    print(f'  Train time: {train_time:.2f}s')
    print(f'  Infer time/sample (val): {infer_per_sample_ms:.3f}ms')

    return {
        'method': method,
        'component_name': component_name,
        'component_op': component_op,
        'X_train_components': X_train_components,
        'X_val_components': X_val_components,
        'train_time_s': train_time,
        'infer_time_per_sample_ms': infer_per_sample_ms,
    }


def transform_components(component_result, X):
    return component_result['component_op'].transform(X)


COMPONENT_METHOD_CONFIGS = [
    {'name': 'kpca_rbf', 'method': 'kpca', 'n_components': max(K_SWEEP_VALUES), 'kernel': 'rbf'},
]
ACTIVE_COMPONENT_METHOD = 'kpca_rbf' 

#### =============================================================================
#### STAGE 2: KMeans Silhouette Component Ranking
#### =============================================================================

In [7]:
def stage_2_kmeans_silhouette_ranking(X_train_components, component_name='Components', top_k=50, n_clusters=4):
    """
    Unsupervised ranking by single-component KMeans cluster-separation score.
    For each component i, run 1D KMeans and score with absolute silhouette.
    """
    print(f'  Ranking {X_train_components.shape[1]} {component_name} components using KMeans silhouette...')

    component_scores = np.zeros(X_train_components.shape[1])

    for i in range(X_train_components.shape[1]):
        xi = X_train_components[:, [i]]
        try:
            kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
            labels = kmeans.fit_predict(xi)
            if len(np.unique(labels)) < 2:
                component_scores[i] = 0.0
            else:
                score = silhouette_score(xi, labels, sample_size=None)
                component_scores[i] = abs(float(score))
        except Exception:
            component_scores[i] = 0.0

    ranked_indices = np.argsort(component_scores)[::-1]
    top_k_eff = min(int(top_k), int(X_train_components.shape[1]))
    top_k_indices = ranked_indices[:top_k_eff].tolist()

    print(f'  Top-{top_k_eff} components (by silhouette score, {component_name}):')
    for i, idx in enumerate(top_k_indices[:5]):
        print(f'    {i+1}. Component {idx}: {component_scores[idx]:.6f}')

    print(f' STAGE 2 ({component_name}): KMeans silhouette ranking complete')

    return {
        'top_k_indices': top_k_indices,
        'component_scores': component_scores,
        'clustering_method': 'kmeans',
    }

#### =============================================================================
#### STAGE 3: Train MLP Feature Selector
#### =============================================================================

In [8]:
class FeatureAttentionBlock(nn.Module):
    def __init__(self, dim, dropout=0.2, se_ratio=0.25):
        super().__init__()
        hidden = max(32, dim // 2)
        se_hidden = max(16, int(dim * se_ratio))

        self.norm = nn.LayerNorm(dim)
        self.ff = nn.Sequential(
            nn.Linear(dim, hidden),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, dim),
            nn.Dropout(dropout),
        )
        self.gate = nn.Sequential(
            nn.Linear(dim, dim),
            nn.Sigmoid(),
        )
        self.channel_attn = nn.Sequential(
            nn.Linear(dim, se_hidden),
            nn.ReLU(),
            nn.Linear(se_hidden, dim),
            nn.Sigmoid(),
        )

    def forward(self, x):
        h = self.norm(x)
        ff_out = self.ff(h)
        gated = ff_out * self.gate(h)
        x = x + gated
        scale = self.channel_attn(x)
        x = x * scale + x
        return x


class AttentionFeatureSelectorMLP(nn.Module):
    def __init__(self, n_input_genes, n_output_components, hidden_dims=None, dropout=0.4):
        super().__init__()
        if hidden_dims is None:
            hidden_dims = [1024, 512, 256, 128]

        self.stem = nn.Sequential(
            nn.Linear(n_input_genes, hidden_dims[0]),
            nn.BatchNorm1d(hidden_dims[0]),
            nn.GELU(),
            nn.Dropout(dropout),
        )

        blocks = []
        transitions = []
        for i, dim in enumerate(hidden_dims):
            blocks.append(FeatureAttentionBlock(dim, dropout=dropout))
            if i < len(hidden_dims) - 1:
                transitions.append(
                    nn.Sequential(
                        nn.Linear(dim, hidden_dims[i + 1]),
                        nn.BatchNorm1d(hidden_dims[i + 1]),
                        nn.GELU(),
                        nn.Dropout(dropout),
                    )
                )
        self.blocks = nn.ModuleList(blocks)
        self.transitions = nn.ModuleList(transitions)

        self.component_head = nn.Linear(hidden_dims[-1], n_output_components)

    def forward(self, x):
        h = self.stem(x)
        for i, block in enumerate(self.blocks):
            h = block(h)
            if i < len(self.transitions):
                h = self.transitions[i](h)
        return self.component_head(h)


def compute_distance_matrix(X_components):
    from sklearn.metrics import pairwise_distances
    return pairwise_distances(X_components, metric='euclidean')


def mine_triplets_from_distances(dist_matrix, n_triplets=512, random_state=42):
    rng = np.random.RandomState(random_state)
    n = dist_matrix.shape[0]
    anchors, positives, negatives = [], [], []

    for _ in range(n_triplets):
        anchor = rng.randint(0, n)
        dists = dist_matrix[anchor].copy()
        dists[anchor] = np.inf

        positive = int(np.argmin(dists))

        median_dist = np.median(dists[dists < np.inf])
        far_candidates = np.where(dists > median_dist)[0]
        if len(far_candidates) == 0:
            far_candidates = np.argsort(dists)[-10:]
        negative = int(rng.choice(far_candidates))

        anchors.append(anchor)
        positives.append(positive)
        negatives.append(negative)

    return np.array(anchors), np.array(positives), np.array(negatives)


def stage_3_train_component_predictor(X_train, X_val, X_train_components, X_val_components,
                                      top_k_indices, epochs=100, batch_size=32, lr=1e-3, patience=10,
                                      y_train_labels=None, y_val_labels=None,
                                      component_loss_weight=1.0, grade_loss_weight=1.0,
                                      random_state=42):
    """
    Attention-based reconstruction model (no joint loss).
    Learns to predict selected top-K components from gene expression.
    """

    random.seed(random_state)
    np.random.seed(random_state)
    torch.manual_seed(random_state)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(random_state)

    component_targets_train = X_train_components[:, top_k_indices]
    component_targets_val = X_val_components[:, top_k_indices]

    from sklearn.preprocessing import StandardScaler as TargetScaler

    target_scaler = TargetScaler()
    component_targets_train = target_scaler.fit_transform(component_targets_train)
    component_targets_val = target_scaler.transform(component_targets_val)

    print(f'  Component targets shape (train): {component_targets_train.shape}')
    print(f'  Component targets shape (val):   {component_targets_val.shape}')
    if grade_loss_weight != 0:
        print(f'  Note: grade_loss_weight={grade_loss_weight} is ignored (joint loss removed).')

    X_train_tensor = torch.FloatTensor(X_train)
    y_train_comp_tensor = torch.FloatTensor(component_targets_train)
    train_dataset = TensorDataset(X_train_tensor, y_train_comp_tensor)
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

    TRIPLET_LOSS_WEIGHT = 0.3
    dist_matrix = compute_distance_matrix(X_train_components)
    anc_idx, pos_idx, neg_idx = mine_triplets_from_distances(
        dist_matrix, n_triplets=1024, random_state=random_state
    )
    triplet_criterion = nn.TripletMarginLoss(margin=1.0, p=2)

    X_val_tensor = torch.FloatTensor(X_val)
    y_val_comp_tensor = torch.FloatTensor(component_targets_val)

    hidden_dims = [1024, 512, 256, 128]
    model = AttentionFeatureSelectorMLP(
        n_input_genes=X_train.shape[1],
        n_output_components=len(top_k_indices),
        hidden_dims=hidden_dims,
        dropout=0.2,
    )
    model.train()

    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=4)
    component_criterion = nn.MSELoss()

    online_train_losses = []
    train_eval_losses = []
    val_component_losses = []

    best_val_loss = float('inf')
    patience_counter = 0
    best_model_state = None

    print(f'  Training attention-based model for {epochs} epochs...')
    t_train = time.time()
    for epoch in range(epochs):
        epoch_online_loss = 0.0

        model.train()
        for batch_X, batch_comp in train_loader:
            optimizer.zero_grad()
            pred_comp = model(batch_X)
            component_loss = component_criterion(pred_comp, batch_comp) * component_loss_weight
            component_loss.backward()
            optimizer.step()
            epoch_online_loss += component_loss.item()

        epoch_online_loss /= len(train_loader)
        online_train_losses.append(epoch_online_loss)

        if len(anc_idx) > 0:
            model.train()
            optimizer.zero_grad()
            anc_t = torch.FloatTensor(X_train[anc_idx])
            pos_t = torch.FloatTensor(X_train[pos_idx])
            neg_t = torch.FloatTensor(X_train[neg_idx])
            pred_anc = model(anc_t)
            pred_pos = model(pos_t)
            pred_neg = model(neg_t)
            triplet_loss = triplet_criterion(pred_anc, pred_pos, pred_neg) * TRIPLET_LOSS_WEIGHT
            triplet_loss.backward()
            optimizer.step()

        model.eval()
        with torch.no_grad():
            train_pred_eval = model(X_train_tensor)
            train_loss_eval = component_criterion(train_pred_eval, y_train_comp_tensor)
            train_eval_loss = float(train_loss_eval.item()) * component_loss_weight
        train_eval_losses.append(train_eval_loss)

        with torch.no_grad():
            val_pred = model(X_val_tensor)
            val_loss = component_criterion(val_pred, y_val_comp_tensor) * component_loss_weight
            val_loss_item = float(val_loss.item())
        val_component_losses.append(val_loss_item)

        if val_loss_item < best_val_loss:
            best_val_loss = val_loss_item
            patience_counter = 0
            best_model_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        else:
            patience_counter += 1

        scheduler.step(val_loss_item)

        if (epoch + 1) % max(1, epochs // 10) == 0:
            print(f'  Epoch {epoch + 1:3d}/{epochs}: train_eval_loss={train_eval_loss:.4f}, val_loss={val_loss_item:.4f}, patience={patience_counter}/{patience}')

        if patience_counter >= patience:
            print(f'  Early stopping at epoch {epoch + 1} (best val loss={best_val_loss:.4f})')
            break

    train_time = time.time() - t_train

    if best_model_state is not None:
        model.load_state_dict(best_model_state)

    result = {
        'model': model,
        'optimizer': optimizer,
        'best_val_loss': best_val_loss,
        'train_time_s': train_time,
        'online_train_losses': online_train_losses,
        'train_losses': train_eval_losses,
        'val_losses': val_component_losses,
        'train_component_losses': train_eval_losses,
        'val_component_losses': val_component_losses,
        'train_grade_losses': [],
        'val_grade_losses': [],
        'best_val_mse': best_val_loss,
        'target_scaler': target_scaler,
    }

    print(' STAGE 3: Attention Reconstruction Model Trained')
    print(f'  Best val component loss: {best_val_loss:.4f}')
    print(f'  Train time: {train_time:.2f}s')

    return result


MANUAL_COMPONENT_LOSS_WEIGHT = 1.0
MANUAL_GRADE_LOSS_WEIGHT = 0.0
MANUAL_STAGE3_RANDOM_STATE = 42

print(f'Manual Stage 3 seed: {MANUAL_STAGE3_RANDOM_STATE}')

Manual Stage 3 seed: 42


#### =============================================================================
#### STAGE 3b: Validate Selector
#### =============================================================================

In [9]:
def stage_3b_validate_selector(X_test, y_test, model, top_k_indices, component_result, target_scaler=None):
    """
    Validate MLP by comparing predicted top-K components against true selected-method components.
    """

    print('Applying MLP to test set...')

    model.eval()
    X_test_tensor = torch.FloatTensor(X_test)
    with torch.no_grad():
        pred_components_test = model(X_test_tensor).numpy()

    X_test_components = transform_components(component_result, X_test)
    true_components_test = X_test_components[:, top_k_indices]

    if target_scaler is not None:
        true_components_test = target_scaler.transform(true_components_test)

    mse = float(np.mean((pred_components_test - true_components_test) ** 2))

    mean_true = np.mean(np.abs(true_components_test), axis=0)
    mean_pred = np.mean(np.abs(pred_components_test), axis=0)
    true_rank = np.argsort(-mean_true)
    pred_rank = np.argsort(-mean_pred)
    corrcoef, pval = spearmanr(true_rank, pred_rank)

    print(f" STAGE 3b: Component Predictor Validation ({component_result['component_name']})")
    print(f'  Test component MSE: {mse:.4f}')
    print(f'  Spearman r (true rank vs predicted rank): {corrcoef:.4f} (p={pval:.4e})')

    return {
        'component_mse': mse,
        'spearman_r': corrcoef,
        'spearman_p': pval,
        'pred_test_components': pred_components_test,
        'true_test_components': true_components_test,
    }

#### =============================================================================
#### STAGE 4: Final XGBoost Predictor
#### =============================================================================

In [10]:
def build_xgb_classifier(scale_pos_weight, random_state=42):
    return xgb.XGBClassifier(
        n_estimators=350,
        max_depth=8,
        learning_rate=0.07,
        subsample=0.8,
        colsample_bytree=0.8,
        min_child_weight=1,
        reg_lambda=5.0,
        gamma=0.0,
        scale_pos_weight=scale_pos_weight,
        max_delta_step=3,
        random_state=random_state,
        n_jobs=1,
        eval_metric='logloss',
        verbose=0,
    )


def class_weights(y):
    n_pos = int((y == 1).sum())
    n_neg = int((y == 0).sum())
    scale_pos_weight = float(n_neg) / float(max(n_pos, 1))
    n_samples = len(y)
    w_neg = n_samples / (2.0 * max(n_neg, 1))
    w_pos = n_samples / (2.0 * max(n_pos, 1))
    sample_weight = np.where(y == 1, w_pos, w_neg)
    return scale_pos_weight, sample_weight


def stage_4_final_xgboost_cv(X_train_scaled, y_train, X_val_scaled, y_val,
                             X_test_scaled, y_test, mlp_model, top_k_indices,
                             n_splits=5, random_state=42):
    """
    Report 1: strict hold-out test performance (train+val -> test).
    Report 2: exploratory 5-fold stratified CV on merged data.
    """

    y_train = y_train.astype(int)
    y_val = y_val.astype(int)
    y_test = y_test.astype(int)

    t0 = time.time()
    mlp_model.eval()
    with torch.no_grad():
        X_train_topk = mlp_model(torch.FloatTensor(X_train_scaled)).numpy()
        X_val_topk = mlp_model(torch.FloatTensor(X_val_scaled)).numpy()
        X_test_topk = mlp_model(torch.FloatTensor(X_test_scaled)).numpy()
    n_total = len(X_train_scaled) + len(X_val_scaled) + len(X_test_scaled)
    mlp_infer_per_sample_ms = (time.time() - t0) / max(n_total, 1) * 1000

    print('  Reproducibility settings: seed=42, deterministic torch=True, xgboost n_jobs=1')

    X_trainval_topk = np.vstack([X_train_topk, X_val_topk])
    y_trainval = np.concatenate([y_train, y_val])
    scale_pos_weight_tv, sample_weight_tv = class_weights(y_trainval)

    xgb_holdout = build_xgb_classifier(scale_pos_weight=scale_pos_weight_tv, random_state=random_state)

    t0 = time.time()
    xgb_holdout.fit(X_trainval_topk, y_trainval, sample_weight=sample_weight_tv)
    holdout_train_time = time.time() - t0

    t0 = time.time()
    y_pred_holdout_test = xgb_holdout.predict(X_test_topk)
    y_pred_proba_holdout_test = xgb_holdout.predict_proba(X_test_topk)
    holdout_infer_per_sample_ms = (time.time() - t0) / max(len(X_test_topk), 1) * 1000

    holdout_accuracy = accuracy_score(y_test, y_pred_holdout_test)
    holdout_balanced_acc = balanced_accuracy_score(y_test, y_pred_holdout_test)
    holdout_f1_weighted = f1_score(y_test, y_pred_holdout_test, average='weighted')
    holdout_cm = confusion_matrix(y_test, y_pred_holdout_test)

    print('  Hold-out test summary (train+val -> test):')
    print(f'    Accuracy:          {holdout_accuracy:.4f}')
    print(f'    Balanced Accuracy: {holdout_balanced_acc:.4f}')
    print(f'    F1 (weighted):     {holdout_f1_weighted:.4f}')
    print(f'    Train time: {holdout_train_time:.2f}s | Infer time/sample: {holdout_infer_per_sample_ms:.3f}ms')

    X_all_topk = np.vstack([X_train_topk, X_val_topk, X_test_topk])
    y_all = np.concatenate([y_train, y_val, y_test])

    print(f'  Merged dataset for CV: X={X_all_topk.shape}, y={y_all.shape}')
    print(f'  Running StratifiedKFold with n_splits={n_splits}, random_state={random_state}')

    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)

    fold_metrics = []
    y_oof_pred = np.zeros_like(y_all)
    y_oof_proba = np.zeros(len(y_all), dtype=float)

    for fold_idx, (train_idx, val_idx) in enumerate(skf.split(X_all_topk, y_all), start=1):
        X_fold_train, X_fold_val = X_all_topk[train_idx], X_all_topk[val_idx]
        y_fold_train, y_fold_val = y_all[train_idx], y_all[val_idx]

        scale_pos_weight, sample_weight = class_weights(y_fold_train)
        xgb_fold = build_xgb_classifier(scale_pos_weight=scale_pos_weight, random_state=random_state)
        xgb_fold.fit(X_fold_train, y_fold_train, sample_weight=sample_weight)

        y_fold_pred = xgb_fold.predict(X_fold_val)
        y_fold_proba = xgb_fold.predict_proba(X_fold_val)[:, 1]

        fold_acc = accuracy_score(y_fold_val, y_fold_pred)
        fold_bal_acc = balanced_accuracy_score(y_fold_val, y_fold_pred)
        fold_f1 = f1_score(y_fold_val, y_fold_pred, average='weighted')

        fold_metrics.append({
            'fold': fold_idx,
            'accuracy': fold_acc,
            'balanced_accuracy': fold_bal_acc,
            'f1_weighted': fold_f1,
        })

        y_oof_pred[val_idx] = y_fold_pred
        y_oof_proba[val_idx] = y_fold_proba

        print(f'  Fold {fold_idx}: Acc={fold_acc:.4f}, BalAcc={fold_bal_acc:.4f}, F1={fold_f1:.4f}')

    oof_accuracy = accuracy_score(y_all, y_oof_pred)
    oof_balanced_acc = balanced_accuracy_score(y_all, y_oof_pred)
    oof_f1_weighted = f1_score(y_all, y_oof_pred, average='weighted')
    oof_cm = confusion_matrix(y_all, y_oof_pred)

    cv_accuracy = float(np.mean([m['accuracy'] for m in fold_metrics]))
    cv_balanced_accuracy = float(np.mean([m['balanced_accuracy'] for m in fold_metrics]))
    cv_f1_weighted = float(np.mean([m['f1_weighted'] for m in fold_metrics]))
    cv_accuracy_std = float(np.std([m['accuracy'] for m in fold_metrics]))
    cv_balanced_accuracy_std = float(np.std([m['balanced_accuracy'] for m in fold_metrics]))
    cv_f1_weighted_std = float(np.std([m['f1_weighted'] for m in fold_metrics]))

    print(f'  CV Summary: Acc={oof_accuracy:.4f}, BalAcc={oof_balanced_acc:.4f}, F1={oof_f1_weighted:.4f}')

    return {
        'X_train_topk': X_train_topk,
        'X_val_topk': X_val_topk,
        'X_test_topk': X_test_topk,
        'X_all_topk': X_all_topk,
        'y_all': y_all,
        'y_pred_test': y_pred_holdout_test,
        'y_pred_proba_test': y_pred_proba_holdout_test,
        'holdout_metrics': {
            'accuracy': holdout_accuracy,
            'balanced_accuracy': holdout_balanced_acc,
            'f1_weighted': holdout_f1_weighted,
            'confusion_matrix': holdout_cm.tolist(),
        },
        'cv_metrics': {
            'accuracy': cv_accuracy,
            'balanced_accuracy': cv_balanced_accuracy,
            'f1_weighted': cv_f1_weighted,
            'accuracy_std': cv_accuracy_std,
            'balanced_accuracy_std': cv_balanced_accuracy_std,
            'f1_weighted_std': cv_f1_weighted_std,
            'oof_accuracy': oof_accuracy,
            'oof_balanced_accuracy': oof_balanced_acc,
            'oof_f1_weighted': oof_f1_weighted,
            'confusion_matrix': oof_cm.tolist(),
            'fold_metrics': fold_metrics,
        },
        'y_oof_pred': y_oof_pred,
        'y_oof_proba': y_oof_proba,
        'mlp_infer_per_sample_ms': mlp_infer_per_sample_ms,
        'holdout_train_time_s': holdout_train_time,
        'holdout_infer_per_sample_ms': holdout_infer_per_sample_ms,
        'settings': {
            'random_state': int(random_state),
            'torch_deterministic': True,
            'xgboost_n_jobs': 1,
            'cv_n_splits': int(n_splits),
        },
    }


def select_and_transform(X_train, y_train, X_eval, score_func, k):
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_eval_scaled = scaler.transform(X_eval)

    k_eff = min(k, X_train_scaled.shape[1])
    selector = SelectKBest(score_func=score_func, k=k_eff)
    X_train_selected = selector.fit_transform(X_train_scaled, y_train)
    X_eval_selected = selector.transform(X_eval_scaled)
    return scaler, selector, X_train_selected, X_eval_selected


def run_feature_selection_baseline(data, feature_names, method_name, score_func, n_splits=5, random_state=42, top_k=50):
    X_train = data['X_train']
    y_train = data['y_train'].astype(int)
    X_val = data['X_val']
    y_val = data['y_val'].astype(int)
    X_test = data['X_test']
    y_test = data['y_test'].astype(int)

    holdout_scaler, holdout_selector, X_train_selected, X_val_selected = select_and_transform(
        X_train, y_train, X_val, score_func, top_k
    )
    X_test_scaled = holdout_scaler.transform(X_test)
    X_test_selected = holdout_selector.transform(X_test_scaled)

    X_trainval_selected = np.vstack([X_train_selected, X_val_selected])
    y_trainval = np.concatenate([y_train, y_val])
    scale_pos_weight, sample_weight = class_weights(y_trainval)

    xgb_holdout = build_xgb_classifier(scale_pos_weight=scale_pos_weight, random_state=random_state)
    xgb_holdout.fit(X_trainval_selected, y_trainval, sample_weight=sample_weight)

    y_pred_test = xgb_holdout.predict(X_test_selected)
    holdout_accuracy = accuracy_score(y_test, y_pred_test)
    holdout_balanced_acc = balanced_accuracy_score(y_test, y_pred_test)
    holdout_f1 = f1_score(y_test, y_pred_test, average='weighted')
    holdout_cm = confusion_matrix(y_test, y_pred_test)

    X_all = np.vstack([X_train, X_val, X_test])
    y_all = np.concatenate([y_train, y_val, y_test])
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)

    y_oof_pred = np.zeros_like(y_all)
    fold_metrics = []

    for fold_idx, (train_idx, val_idx) in enumerate(skf.split(X_all, y_all), start=1):
        X_fold_train, X_fold_val = X_all[train_idx], X_all[val_idx]
        y_fold_train, y_fold_val = y_all[train_idx], y_all[val_idx]

        scale_pos_weight_f, sample_weight_f = class_weights(y_fold_train)
        _, selector_f, X_fold_train_selected, X_fold_val_selected = select_and_transform(
            X_fold_train, y_fold_train, X_fold_val, score_func, top_k
        )

        xgb_fold = build_xgb_classifier(scale_pos_weight=scale_pos_weight_f, random_state=random_state)
        xgb_fold.fit(X_fold_train_selected, y_fold_train, sample_weight=sample_weight_f)

        y_fold_pred = xgb_fold.predict(X_fold_val_selected)
        y_oof_pred[val_idx] = y_fold_pred

        fold_metrics.append({
            'fold': fold_idx,
            'accuracy': accuracy_score(y_fold_val, y_fold_pred),
            'balanced_accuracy': balanced_accuracy_score(y_fold_val, y_fold_pred),
            'f1_weighted': f1_score(y_fold_val, y_fold_pred, average='weighted'),
        })

    cv_accuracy = float(np.mean([m['accuracy'] for m in fold_metrics]))
    cv_balanced_accuracy = float(np.mean([m['balanced_accuracy'] for m in fold_metrics]))
    cv_f1_weighted = float(np.mean([m['f1_weighted'] for m in fold_metrics]))
    cv_accuracy_std = float(np.std([m['accuracy'] for m in fold_metrics]))
    cv_balanced_accuracy_std = float(np.std([m['balanced_accuracy'] for m in fold_metrics]))
    cv_f1_weighted_std = float(np.std([m['f1_weighted'] for m in fold_metrics]))

    results = {
        'method': method_name,
        'top_k': min(top_k, len(feature_names)),
        'holdout_accuracy': holdout_accuracy,
        'holdout_balanced_accuracy': holdout_balanced_acc,
        'holdout_f1_weighted': holdout_f1,
        'holdout_confusion_matrix': holdout_cm.tolist(),
        'cv_accuracy': cv_accuracy,
        'cv_balanced_accuracy': cv_balanced_accuracy,
        'cv_f1_weighted': cv_f1_weighted,
        'cv_accuracy_std': cv_accuracy_std,
        'cv_balanced_accuracy_std': cv_balanced_accuracy_std,
        'cv_f1_weighted_std': cv_f1_weighted_std,
        'cv_confusion_matrix': confusion_matrix(y_all, y_oof_pred).tolist(),
        'fold_metrics': fold_metrics,
        'reproducibility': {
            'seed': random_state,
            'xgboost_n_jobs': 1,
            'cv_n_splits': n_splits,
            'feature_selection_inside_each_fold': True,
        },
    }

    print(f'[{method_name}] Hold-out accuracy={holdout_accuracy:.4f}, balanced_accuracy={holdout_balanced_acc:.4f}, f1={holdout_f1:.4f}')
    print(f'[{method_name}] 5-fold CV balanced_accuracy={cv_balanced_accuracy:.4f} +/- {cv_balanced_accuracy_std:.4f}')

    return results

#### =============================================================================
#### K-SWEEP: OmicSieve + Compact Baselines
#### =============================================================================

In [11]:
FEATURE_SELECTION_METHODS = {
    'ANOVA + XGBoost': f_classif,
    'Mutual Information + XGBoost': partial(mutual_info_classif, random_state=42),
}

genesieve_sweep_results = []
baseline_sweep_results = []

for k in K_SWEEP_VALUES:
    print()
    print('=' * 90)
    print(f'K SWEEP ITERATION: k = {k}')
    print('=' * 90)

    # Stage 1
    component_result = stage_1_unsupervised_components(
        data['X_train'],
        data['X_val'],
        method='kpca',
        n_components=max(K_SWEEP_VALUES),
        kernel='rbf',
    )

    # Stage 2
    shap_result = stage_2_kmeans_silhouette_ranking(
        component_result['X_train_components'],
        component_name=component_result['component_name'],
        top_k=k,
        n_clusters=4,
    )

    # Stage 3
    mlp_result = stage_3_train_component_predictor(
        data['X_train'],
        data['X_val'],
        component_result['X_train_components'],
        component_result['X_val_components'],
        shap_result['top_k_indices'],
        epochs=200,
        batch_size=32,
        lr=1e-3,
        patience=20,
        y_train_labels=data['y_train'],
        y_val_labels=data['y_val'],
        component_loss_weight=MANUAL_COMPONENT_LOSS_WEIGHT,
        grade_loss_weight=MANUAL_GRADE_LOSS_WEIGHT,
        random_state=MANUAL_STAGE3_RANDOM_STATE,
    )

    # Stage 3b
    validation_result = stage_3b_validate_selector(
        data['X_test'],
        data['y_test'],
        mlp_result['model'],
        shap_result['top_k_indices'],
        component_result,
        target_scaler=mlp_result['target_scaler'],
    )

    # Stage 4
    xgb_final_result = stage_4_final_xgboost_cv(
        data['X_train'],
        data['y_train'],
        data['X_val'],
        data['y_val'],
        data['X_test'],
        data['y_test'],
        mlp_result['model'],
        shap_result['top_k_indices'],
        n_splits=5,
        random_state=42,
    )

    genesieve_sweep_results.append({
        'task': task,
        'method': 'OmicSieve',
        'k': int(k),
        'holdout_balanced_accuracy': float(xgb_final_result['holdout_metrics']['balanced_accuracy']),
        'cv_balanced_accuracy': float(xgb_final_result['cv_metrics']['balanced_accuracy']),
        'cv_balanced_accuracy_std': float(xgb_final_result['cv_metrics']['balanced_accuracy_std']),
        'holdout_accuracy': float(xgb_final_result['holdout_metrics']['accuracy']),
        'cv_accuracy': float(xgb_final_result['cv_metrics']['accuracy']),
        'cv_accuracy_std': float(xgb_final_result['cv_metrics']['accuracy_std']),
        'selector_component_mse': float(validation_result['component_mse']),
        'selector_spearman_r': float(validation_result['spearman_r']),
    })

    # Compact feature-selection baselines
    for method_name, score_func in FEATURE_SELECTION_METHODS.items():
        baseline_result = run_feature_selection_baseline(
            data=data,
            feature_names=feature_names,
            method_name=method_name,
            score_func=score_func,
            n_splits=5,
            random_state=42,
            top_k=k,
        )

        baseline_sweep_results.append({
            'task': task,
            'method': method_name,
            'k': int(k),
            'holdout_balanced_accuracy': float(baseline_result['holdout_balanced_accuracy']),
            'cv_balanced_accuracy': float(baseline_result['cv_balanced_accuracy']),
            'cv_balanced_accuracy_std': float(baseline_result['cv_balanced_accuracy_std']),
            'holdout_accuracy': float(baseline_result['holdout_accuracy']),
            'cv_accuracy': float(baseline_result['cv_accuracy']),
            'cv_accuracy_std': float(baseline_result['cv_accuracy_std']),
        })

all_results = genesieve_sweep_results + baseline_sweep_results
results_df = pd.DataFrame(all_results)
results_df = results_df.sort_values(['method', 'k']).reset_index(drop=True)

results_df[['method', 'k', 'holdout_balanced_accuracy', 'cv_balanced_accuracy', 'cv_balanced_accuracy_std']]


K SWEEP ITERATION: k = 10
  [KPCA] X_train shape: (1745, 19310)
STAGE 1: KPCA_RBF fitted
  X_train_components: (1745, 100)
  X_val_components: (374, 100)
  Train time: 0.72s
  Infer time/sample (val): 0.339ms
  Ranking 100 KPCA_RBF components using KMeans silhouette...
  Top-10 components (by silhouette score, KPCA_RBF):
    1. Component 1: 0.586171
    2. Component 0: 0.576294
    3. Component 5: 0.573006
    4. Component 7: 0.563127
    5. Component 3: 0.562900
 STAGE 2 (KPCA_RBF): KMeans silhouette ranking complete
  Component targets shape (train): (1745, 10)
  Component targets shape (val):   (374, 10)
  Training attention-based model for 200 epochs...
  Epoch  20/200: train_eval_loss=0.0286, val_loss=0.0343, patience=0/20
  Epoch  40/200: train_eval_loss=0.0191, val_loss=0.0257, patience=3/20
  Epoch  60/200: train_eval_loss=0.0102, val_loss=0.0207, patience=11/20
  Epoch  80/200: train_eval_loss=0.0104, val_loss=0.0211, patience=13/20
  Epoch 100/200: train_eval_loss=0.0098, va

,method,k,holdout_balanced_accuracy,cv_balanced_accuracy,cv_balanced_accuracy_std
0,ANOVA + XGBoost,10,0.753470,0.745520,0.035551
1,ANOVA + XGBoost,20,0.736832,0.757864,0.029284
2,ANOVA + XGBoost,30,0.721653,0.758636,0.030666
3,ANOVA + XGBoost,50,0.758108,0.766395,0.030563
4,ANOVA + XGBoost,80,0.791580,0.768696,0.036829
5,ANOVA + XGBoost,100,0.797678,0.768261,0.033991
6,OmicSieve,10,0.800629,0.762527,0.037918
7,OmicSieve,20,0.793072,0.792366,0.041868
8,OmicSieve,30,0.815906,0.803871,0.044804
9,OmicSieve,50,0.869129,0.829903,0.039652


#### =============================================================================
#### Final Balanced-Accuracy Plot (Plotly)
#### =============================================================================

In [12]:
method_styles = {
    'OmicSieve': {'color': '#1f77b4', 'symbol': 'circle'},
    'ANOVA + XGBoost': {'color': '#ff7f0e', 'symbol': 'square'},
    'Mutual Information + XGBoost': {'color': '#2ca02c', 'symbol': 'diamond'},
}

fig = go.Figure()

for method_name, style in method_styles.items():
    method_df = results_df[results_df['method'] == method_name].sort_values('k')
    if method_df.empty:
        continue

    fig.add_trace(
        go.Scatter(
            x=method_df['k'],
            y=method_df['holdout_balanced_accuracy'],
            mode='lines+markers',
            name=f'{method_name} (Hold-out)',
            line=dict(color=style['color'], width=3),
            marker=dict(symbol=style['symbol'], size=9, line=dict(width=1, color='white')),
            hovertemplate=(
                'Method: %{customdata[0]}<br>'
                'k: %{x}<br>'
                'Hold-out Balanced Acc: %{y:.4f}<extra></extra>'
            ),
            customdata=np.column_stack([method_df['method']]),
        )
    )

    if INCLUDE_CV_CURVES:
        fig.add_trace(
            go.Scatter(
                x=method_df['k'],
                y=method_df['cv_balanced_accuracy'],
                mode='lines+markers',
                name=f'{method_name} (5-Fold CV)',
                line=dict(color=style['color'], width=2, dash='dash'),
                marker=dict(symbol=style['symbol'], size=7),
                error_y=dict(
                    type='data',
                    array=method_df['cv_balanced_accuracy_std'],
                    visible=True,
                    thickness=1.2,
                    width=4,
                ),
                opacity=0.85,
                hovertemplate=(
                    'Method: %{customdata[0]}<br>'
                    'k: %{x}<br>'
                    'CV Balanced Acc: %{y:.4f}<br>'
                    'CV Std: %{customdata[1]:.4f}<extra></extra>'
                ),
                customdata=np.column_stack([method_df['method'], method_df['cv_balanced_accuracy_std']]),
            )
        )

fig.update_layout(
    title=f'OmicSieve vs Compact Feature Selection Baselines ({task.upper()})<br><sup>Balanced Accuracy Across Selected Components/Features (k)</sup>',
    template='plotly_white',
    width=1100,
    height=650,
    hovermode='x unified',
    legend=dict(
        title='Method',
        orientation='h',
        yanchor='bottom',
        y=1.02,
        xanchor='left',
        x=0.0,
    ),
    margin=dict(l=70, r=30, t=110, b=70),
)

fig.update_xaxes(
    title='Number of selected components/features (k)',
    tickmode='array',
    tickvals=K_SWEEP_VALUES,
    showline=True,
    linewidth=1.5,
    linecolor='#2b2b2b',
    mirror=True,
    gridcolor='rgba(0, 0, 0, 0.08)',
)

fig.update_yaxes(
    title='Balanced Accuracy',
    range=[0.45, 1.0],
    tickformat='.2f',
    showline=True,
    linewidth=1.5,
    linecolor='#2b2b2b',
    mirror=True,
    gridcolor='rgba(0, 0, 0, 0.08)',
)

fig.write_html(FINAL_FIG_PATH, include_plotlyjs='cdn')
print(f'Saved final Plotly figure to: {FINAL_FIG_PATH.resolve()}')

fig

Saved final Plotly figure to: /home/amir_ebrahimi/Documents/OmicSieve/OmicSieve/Code/grade_k_sweep_balanced_accuracy_plotly.html
